# Completion run: depth control, real data, integrity

**Paper:** *Where the Dimension Hides: The Approximation Constant of Kolmogorov–Arnold Networks*
**Author:** Haydar Kılıç · ORCID 0000-0002-2551-3772

Three items remained open on the code side after the main study and the E1–E3 follow-ups.
This notebook closes all three. Run all cells.

| Part | What it does | Which gap it closes | Cost |
|---|---|---|---|
| **E4** | Fits the `WidthKAN` probe, which varies the edge count $E = md$ at **depth one** | The depth confound. `EdgeKAN` vs `KANet` is currently confounded with depth as well as edge count; this is the last rival explanation for the paper's central finding | 2–4 GPU-h |
| **R** | Real-data probe fits and controls C5/C6, with the budget grid adapted to each dataset | Table 7's real rows and §4.10 are in the paper but not reproducible from the released notebook; and five of six real cells in C5/C6 returned "unavailable" | ~1 GPU-h |
| **I** | The three equivalence checks reported in §4.1 | The values are in `integrity.json` but the clean notebook never recomputed them | seconds |

### Why E4 matters most

The paper's central empirical finding is that `KANet` shows a dimensional slope of about
$+0.7$ (train) to $+0.9$ (test) at fixed per-edge resolution, where the edge-law premise
requires zero. Three rival explanations have been eliminated: the edge count (the additive
probe and the oracle both sit near zero), over-parametrisation (E1), and generalisation failure
(the slope is present on the training split). **Depth has not been eliminated.**

`WidthKAN` is depth one and varies $E$ through its output width, with a fixed non-trainable
read-out. Its fixed-resolution slope therefore separates the two:

* slope $\approx 0$, like `EdgeKAN` — the effect is specific to the two-layer *composition*;
* slope $\approx +1$, like `KANet` — the effect follows the *edge count*, not depth;
* in between — both contribute.

Any of the three narrows §5.2, which currently reports a measured effect with no identified
cause.

## 0. Setup

In [ ]:
import os, json, math, time, random, itertools, hashlib
import numpy as np
import pandas as pd
from scipy import stats
import torch
import torch.nn as nn
import torch.nn.functional as F

SCALE = "full"          # "smoke" (CPU, minutes) | "trim" (GPU, ~30 min) | "full" (GPU, 3-5 h)

REQUIRE_PERSISTENT = True    # refuse to run if results would land on ephemeral storage

def _resolve_out():
    """Resolve the output directory, and refuse to proceed if it is not persistent.

    Colab wipes /content when the runtime is recycled. An earlier version of this notebook
    fell back to /content silently when Drive was unavailable, and a completed run was lost
    that way. It now fails loudly instead.
    """
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        d = "/content/drive/MyDrive/KAN_JMLR/completion_outputs"
        os.makedirs(d, exist_ok=True)
        probe = os.path.join(d, ".write_test")
        with open(probe, "w") as f: f.write("ok")
        os.remove(probe)
        print(f"[storage] persistent: {d}")
        return d
    except ImportError:
        d = os.path.abspath("./completion_outputs")
        print(f"[storage] not on Colab; writing locally to {d}")
        return d
    except Exception as e:
        msg = ("Google Drive could not be mounted (" + type(e).__name__ + "). Results would "
               "be written to /content, which Colab deletes when the runtime is recycled -- "
               "a completed run has already been lost this way.\n"
               "Either mount Drive and re-run this cell, or set REQUIRE_PERSISTENT = False "
               "to accept the risk and remember to download the CSVs before closing.")
        if REQUIRE_PERSISTENT:
            raise RuntimeError(msg)
        print("[storage] WARNING -- " + msg)
        return os.path.abspath("/content/completion_outputs")

OUT = _resolve_out()
os.makedirs(OUT, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.use_deterministic_algorithms(True, warn_only=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

SCALES = {
    "smoke": dict(dims=[2, 4, 8], orders=[3], grids=[4, 8, 16], widths=[1, 2, 4],
                  seeds=[0], epochs=40, n_base=2_000, real=["synthetic", "airfoil"]),
    "trim":  dict(dims=[2, 4, 8, 16], orders=[2, 3], grids=[4, 8, 16, 32],
                  widths=[1, 2, 4, 8, 16], seeds=[0, 1], epochs=200, n_base=8_000,
                  real=["synthetic", "airfoil", "concrete", "energy"]),
    "full":  dict(dims=[2, 4, 8, 16, 32], orders=[2, 3, 4], grids=[4, 8, 16, 32, 48],
                  widths=[1, 2, 4, 8, 16], seeds=[0, 1, 2], epochs=500, n_base=16_000,
                  real=["synthetic", "airfoil", "concrete", "energy", "wine",
                        "superconduct"]),
}
S = SCALES[SCALE]

BATCH, LR, WEIGHT_DECAY, CAP_RATIO = 2_048, 3e-3, 1e-5, 0.25
MAX_HOURS  = 6.0
TIME_SHARE = {"E4": 0.75, "R": 0.25}

class Budget:
    'Graceful stop after a fixed share of the wall-clock budget.'
    def __init__(self, name):
        self.name, self.limit, self.t0, self.stopped = name, MAX_HOURS*3600*TIME_SHARE[name], time.time(), False
    def exhausted(self):
        if time.time() - self.t0 > self.limit and not self.stopped:
            self.stopped = True
            print(f"  [{self.name}] time share of {self.limit/3600:.1f} h reached -- stopping "
                  f"gracefully; re-run to continue.")
        return self.stopped

def _load(csv, label):
    'Read a result file, returning an empty frame if it is missing.'
    if not os.path.exists(csv):
        print(f"[{label}] no results yet at {csv}"); return pd.DataFrame()
    return pd.read_csv(csv)

print(f"scale = {SCALE}   device = {DEVICE}\noutputs -> {OUT}")
if DEVICE.type == "cpu" and SCALE != "smoke":
    print("\n*** WARNING: no GPU and SCALE is not 'smoke'. This will take days. ***\n")

## 1. Primitives

Reproduced exactly as in the main pipeline so that results are directly comparable. If you
change anything here, change it there too.

In [ ]:
class KANLayer(nn.Module):
    'B-spline KAN layer: (G+k) coefficients + 1 residual weight per edge.'
    def __init__(self, in_dim, out_dim, grid=8, order=3, grid_range=(-1., 1.)):
        super().__init__()
        self.order, self.grid, self.grid_range = order, grid, grid_range
        g0, g1 = grid_range
        step = (g1 - g0) / grid
        self.register_buffer("knots", torch.arange(-order, grid + order + 1) * step + g0)
        self.coeff = nn.Parameter(torch.randn(in_dim, out_dim, grid + order) * 0.1)
        self.w_res = nn.Parameter(torch.randn(in_dim, out_dim) * 0.1)

    def _basis(self, x):
        g0, g1 = self.grid_range
        eps = 1e-6 * (g1 - g0)
        x = x.clamp(g0 + eps, g1 - eps).unsqueeze(-1)
        t = self.knots
        b = ((x >= t[:-1]) & (x < t[1:])).to(x.dtype)
        for k in range(1, self.order + 1):
            ld = (t[k:-1] - t[:-(k + 1)]).clamp_min(1e-8)
            rd = (t[k + 1:] - t[1:-k]).clamp_min(1e-8)
            b = (x - t[:-(k + 1)]) / ld * b[..., :-1] + (t[k + 1:] - x) / rd * b[..., 1:]
        return b

    def forward(self, x):
        return (torch.einsum("bik,iok->bio", self._basis(x), self.coeff)
                + F.silu(x).unsqueeze(-1) * self.w_res).sum(1)


class EdgeKAN(nn.Module):
    'Additive probe, depth 1.  E = d.'
    def __init__(self, d, grid, order):
        super().__init__(); self.kan = KANLayer(d, 1, grid=grid, order=order)
    def forward(self, x): return self.kan(x)


class KANet(nn.Module):
    'Two-layer Kolmogorov-Arnold probe, H = 2d+1.  E = (2d+1)(d+1).'
    def __init__(self, d, grid, order):
        super().__init__()
        H = 2 * d + 1
        self.l1 = KANLayer(d, H, grid=grid, order=order)
        self.l2 = KANLayer(H, 1, grid=grid, order=order)
    def forward(self, x): return self.l2(self.l1(x))


class WidthKAN(nn.Module):
    """Depth-1 probe with edge count E = m*d.

    The read-out is a FIXED uniform mix, registered as a buffer rather than a parameter, so
    sweeping m adds edges without adding any free parameter outside the edge accounting of
    Lemma 1. This is what separates the edge-count effect from the effect of depth.
    """
    def __init__(self, d, m, grid, order):
        super().__init__()
        self.kan = KANLayer(d, m, grid=grid, order=order)
        self.register_buffer("mix", torch.ones(m) / m)
    def forward(self, x): return (self.kan(x) * self.mix).sum(1, keepdim=True)


class MLP(nn.Module):
    def __init__(self, d, width, depth=2):
        super().__init__()
        self.inp = nn.Linear(d, width)
        self.blocks = nn.ModuleList([nn.ModuleList(
            [nn.LayerNorm(width), nn.Linear(width, width), nn.Linear(width, width)])
            for _ in range(depth)])
        self.head = nn.Linear(width, 1)
    def forward(self, x):
        h = self.inp(x)
        for ln, a, b in self.blocks:
            h = h + b(F.gelu(a(ln(h))))
        return self.head(h)


class FullKAN(nn.Module):
    def __init__(self, d, width, grid, order):
        super().__init__()
        self.inp = nn.Linear(d, width); self.norm = nn.LayerNorm(width)
        self.kan = KANLayer(width, width, grid=grid, order=order)
        self.head = nn.Linear(width, 1)
    def forward(self, x):
        h = self.inp(x)
        return self.head(h + self.kan(self.norm(h)))


count_params = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
relative_l2  = lambda p, y: (torch.linalg.norm(p - y) /
                             torch.linalg.norm(y).clamp_min(1e-12)).item()

def edge_count(kind, d, m=None):
    if kind in ("edge_kan", "best_spline"): return d
    if kind == "kanet":    return (2 * d + 1) * (d + 1)
    if kind == "width_kan": return m * d
    return None

def _closest_width(a, b, budget, wmax):
    disc = b * b - 4 * a * (1 - budget)
    w0 = (-b + math.sqrt(max(disc, 0.0))) / (2 * a) if a > 0 else (budget - 1) / max(b, 1e-9)
    cands = [w for w in {int(math.floor(w0)) - 1, int(math.floor(w0)),
                         int(math.ceil(w0)), int(math.ceil(w0)) + 1} if 2 <= w < wmax] or [2]
    return min(cands, key=lambda w: abs(a * w * w + b * w + 1 - budget))

full_kan_for_budget = lambda d, N, G, k: FullKAN(d, _closest_width(G + k + 1, d + 4, N, 1024), G, k)
mlp_for_budget      = lambda d, N, L=2: MLP(d, _closest_width(2 * L, 4 * L + d + 2, N, 2048), L)


def _additive(X):
    psi = lambda t: torch.sin(math.pi * t) + 0.3 * torch.cos(3 * math.pi * t)
    return psi(X).sum(1, keepdim=True) / math.sqrt(X.shape[1])

def make_synth(kind, d, n_tr, n_te, seed):
    g = torch.Generator().manual_seed(seed + 1000)
    Xtr = torch.rand(n_tr, d, generator=g) * 2 - 1
    Xte = torch.rand(n_te, d, generator=g) * 2 - 1
    fn = _additive
    ytr, yte = fn(Xtr), fn(Xte)
    mu, sd = ytr.mean(), ytr.std().clamp_min(1e-8)
    to = lambda t: t.to(DEVICE, torch.float32)
    return to(Xtr), to((ytr - mu) / sd), to(Xte), to((yte - mu) / sd)


def train_eval(model, data, epochs, seed, lr=LR, batch=BATCH):
    seed_everything(seed)
    Xtr, ytr, Xte, yte = data
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    n = Xtr.shape[0]
    g = torch.Generator(device="cpu").manual_seed(seed)
    model.train(); last = float("nan")
    for _ in range(epochs):
        perm = torch.randperm(n, generator=g).to(Xtr.device)
        tot, nb = 0.0, 0
        for i in range(0, n, batch):
            idx = perm[i:i + batch]
            opt.zero_grad(set_to_none=True)
            loss = F.mse_loss(model(Xtr[idx]), ytr[idx])
            loss.backward(); opt.step()
            tot += loss.item(); nb += 1
        sch.step(); last = tot / max(nb, 1)
    model.eval()
    with torch.no_grad():
        ch = lambda X: torch.cat([model(X[i:i + 8192]) for i in range(0, X.shape[0], 8192)])
        return dict(rel_l2=relative_l2(ch(Xte), yte), rel_l2_train=relative_l2(ch(Xtr), ytr),
                    train_mse=last)

print("primitives ready")

## 2. Part E4 — the fitted `WidthKAN` sweep

The protocol matches E1 exactly, so the results are directly comparable: the sample size is
tied to the budget, $n_{\mathrm{tr}} = \max(n_0, \lceil N/0.25\rceil)$, and the target is the
additive function that `WidthKAN` can represent.

The quantity of interest is the **dimensional slope at fixed resolution**. The edge-law premise
predicts zero for any architecture whose error is governed by per-edge resolution alone.
`EdgeKAN` satisfies it; `KANet` does not. `WidthKAN` sits between them in edge count but shares
`EdgeKAN`'s depth.

In [ ]:
def run_E4(S, out_csv):
    done = set()
    if os.path.exists(out_csv):
        done = set(pd.read_csv(out_csv)["key"]); print(f"resuming E4: {len(done)} cells present")
    B, rows, t0 = Budget("E4"), [], time.time()

    def cost(j):
        m, k, d, G, s = j
        return max(S["n_base"], math.ceil(m * d * (G + k + 1) / CAP_RATIO))
    jobs = sorted(itertools.product(S["widths"], S["orders"], S["dims"], S["grids"], S["seeds"]),
                  key=cost)
    print(f"E4: {len(jobs)} cells, n_train from {cost(jobs[0]):,} to {cost(jobs[-1]):,}")

    for i, (m, k, d, G, s) in enumerate(jobs):
        if B.exhausted(): break
        key = f"E4|m{m}|k{k}|d{d}|G{G}|s{s}"
        if key in done: continue
        E = edge_count("width_kan", d, m)
        params = E * (G + k + 1)
        n_tr = max(S["n_base"], math.ceil(params / CAP_RATIO))
        data = make_synth("additive", d, n_tr, n_tr // 2, s)
        seed_everything(s)
        r = train_eval(WidthKAN(d, m, G, k), data, S["epochs"], s)
        rows.append(dict(key=key, exp="E4", kind="width_kan", m=m, k=k, d=d, grid=G, seed=s,
                         edges=E, params=params, n_train=n_tr,
                         cap_ratio=round(params / n_tr, 4), **r))
        if len(rows) % 25 == 0:
            pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                      header=not os.path.exists(out_csv))
            rows = []
            print(f"  {i+1}/{len(jobs)} cells, {(time.time()-t0)/60:.1f} min")
    if rows:
        pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                  header=not os.path.exists(out_csv))
    print(f"E4 done in {(time.time()-t0)/60:.1f} min")


E4_CSV = os.path.join(OUT, f"E4_widthkan_{SCALE}.csv")
run_E4(S, E4_CSV)

In [ ]:
def dslope_fixed_grid(df, k=None, value="rel_l2", by=None):
    'Slope of log(error) on log(d) at each fixed grid. The premise predicts 0.'
    if df.empty: return pd.DataFrame()
    if k is not None: df = df[df.k == k]
    keys = ["grid"] + ([by] if by else [])
    out = {}
    for kk, g in (df.groupby(by) if by else [("all", df)]):
        ag = g.groupby(["grid", "d"])[value].mean().reset_index()
        row = {}
        for G, gg in ag.groupby("grid"):
            gg = gg.sort_values("d")
            if len(gg) >= 3 and (gg[value] > 0).all():
                row[int(G)] = round(float(np.polyfit(np.log(gg.d), np.log(gg[value]), 1)[0]), 2)
        out[kk] = row
    return pd.DataFrame(out).T.sort_index()


def joint_fit(cells, k):
    N = cells["params"].values.astype(float); E = cells["edges"].values.astype(float)
    y = np.log(cells["err"].values)
    XA = np.column_stack([np.ones(len(N)), np.log(N), np.log(E)])
    cA = np.linalg.lstsq(XA, y, rcond=None)[0]
    Gx = N / E - (k + 1); ok = Gx > 0
    cG = (np.linalg.lstsq(np.column_stack([np.ones(ok.sum()), np.log(Gx[ok])]),
                          y[ok], rcond=None)[0][1] if ok.sum() >= 4 else np.nan)
    return float(cA[1]), float(cA[2]), float(cG)


E4 = _load(E4_CSV, "E4")
if not E4.empty:
    kk = 3 if 3 in E4.k.unique() else int(E4.k.iloc[0])
    print(f"E4 -- dimensional slope at fixed resolution (k={kk}); premise predicts 0\n")
    print("  test split:");  print(dslope_fixed_grid(E4, kk, "rel_l2").to_string())
    print("\n  train split:"); print(dslope_fixed_grid(E4, kk, "rel_l2_train").to_string())
    print("\n  by output width m (test split):")
    print(dslope_fixed_grid(E4, kk, "rel_l2", by="m").to_string())

    print("\nE4 -- joint edge-law fit, both parametrisations:")
    rows = []
    for k, g in E4.groupby("k"):
        cells = (g.groupby(["d", "grid", "params", "edges"])["rel_l2"].mean()
                  .reset_index().rename(columns={"rel_l2": "err"}))
        cells = cells[(cells.err < 0.6) & (cells.err > 0)]
        if len(cells) < 4: continue
        cN, cE, cG = joint_fit(cells, k)
        rows.append(dict(k=int(k), n_cells=len(cells), cN=round(cN, 3), cE=round(cE, 3),
                         cE_pred=k + 1, cG=round(cG, 3), cG_pred=-(k + 1),
                         cG_pass=bool(abs(cG + (k + 1)) <= 0.15 * (k + 1))))
    E4_FIT = pd.DataFrame(rows)
    print(E4_FIT.to_string(index=False))
else:
    E4_FIT = pd.DataFrame()

## 3. Part R — the real-data pipeline

Two things are done here. First, the probe fits on real datasets that produce the *Real* rows
of the joint-fit table. Second, controls C5 and C6, with the fix that the original run needed:
**the budget grid is derived from each dataset's own training split** rather than fixed at
$\{500,\dots,16000\}$. That fixed grid demanded up to $16{,}000$ parameters from datasets whose
training split is under $800$ rows, which is why five of six real cells returned "unavailable".

In [ ]:
DATA_CACHE = os.path.join(OUT, "uci_cache")
os.makedirs(DATA_CACHE, exist_ok=True)
UCI_IDS = dict(airfoil=291, concrete=165, energy=242, wine=186, superconduct=464)


def load_uci(name, seed=0, test_frac=0.3):
    """Fetch (or read from cache) a dataset, z-scored on training statistics.

    The name "synthetic" returns the representable additive target used as the control in
    C5 and C6: it must exhibit the predicted signature, so it validates the machinery
    independently of whether any real dataset is reachable.

    If a real dataset cannot be downloaded, drop a CSV named <name>.csv into the cache
    directory (features first, target last) and it will be picked up.
    """
    if name == "synthetic":
        d, n = 6, 4_000
        g = torch.Generator().manual_seed(seed + 500)
        X = torch.rand(n, d, generator=g) * 2 - 1
        y = _additive(X)
        ntr = int(n * (1 - test_frac))
        mu, sd = y[:ntr].mean(), y[:ntr].std().clamp_min(1e-8)
        to = lambda t: t.to(DEVICE, torch.float32)
        return (to(X[:ntr]), to((y[:ntr] - mu) / sd),
                to(X[ntr:]), to((y[ntr:] - mu) / sd))
    path = os.path.join(DATA_CACHE, f"{name}.csv")
    if not os.path.exists(path):
        try:
            from ucimlrepo import fetch_ucirepo
        except ImportError:
            import subprocess, sys
            for extra in ([], ["--break-system-packages"]):
                r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                                    "ucimlrepo"] + extra)
                if r.returncode == 0: break
            from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=UCI_IDS[name])
        df = pd.concat([ds.data.features, ds.data.targets], axis=1)
        df.to_csv(path, index=False)
    df = pd.read_csv(path).select_dtypes(include=[np.number]).dropna()
    X = df.iloc[:, :-1].values.astype(np.float32)
    y = df.iloc[:, -1:].values.astype(np.float32)
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(X)); ntr = int(len(X) * (1 - test_frac))
    tr, te = idx[:ntr], idx[ntr:]
    mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-8
    ym, ys = y[tr].mean(), y[tr].std() + 1e-8
    to = lambda a: torch.tensor(a, dtype=torch.float32, device=DEVICE)
    # inputs are squashed into [-1, 1], the domain on which the spline grid is defined
    sq = lambda a: np.tanh((a - mu) / sd / 3.0)
    return to(sq(X[tr])), to((y[tr] - ym) / ys), to(sq(X[te])), to((y[te] - ym) / ys)


def append_edges(Xtr, Xte, n_extra, mode, seed):
    """Append spurious coordinates: pure noise, or near-duplicates of existing features."""
    if n_extra == 0: return Xtr, Xte
    g = torch.Generator(device="cpu").manual_seed(seed + 7)
    d = Xtr.shape[1]
    if mode == "noise":
        f = lambda X: torch.cat([X, (torch.rand(X.shape[0], n_extra, generator=g) * 2 - 1
                                     ).to(X.device)], 1)
    else:                                   # duplicate: copy a feature and jitter it slightly
        cols = torch.randint(0, d, (n_extra,), generator=g)
        f = lambda X: torch.cat([X, X[:, cols] + 0.01 * (torch.rand(
            X.shape[0], n_extra, generator=g).to(X.device) * 2 - 1)], 1)
    return f(Xtr), f(Xte)


def dataset_budget_grid(n_tr, k=3, n_points=5):
    """Budget grid adapted to the dataset.

    The original run used a fixed grid up to 16,000 parameters, which exceeds the training
    split of most of these datasets by an order of magnitude; every such cell is deeply
    over-parametrised and the damage signal is swamped. Here the grid spans
    n_tr/32 to n_tr/2 parameters, so the largest cell still satisfies params/n_tr <= 0.5.
    """
    lo, hi = max(50, n_tr // 32), max(100, n_tr // 2)
    return sorted({int(round(v)) for v in np.geomspace(lo, hi, n_points)})


def run_real(S, out_csv):
    done = set()
    if os.path.exists(out_csv):
        done = set(pd.read_csv(out_csv)["key"]); print(f"resuming R: {len(done)} rows present")
    B, rows, t0 = Budget("R"), [], time.time()

    for name in S["real"]:
        try:
            Xtr, ytr, Xte, yte = load_uci(name)
        except Exception as e:
            print(f"  [{name}] could not be loaded ({type(e).__name__}). If the machine has "
                  f"no network access, place {name}.csv in {DATA_CACHE} "
                  f"(features first, target last) and re-run.")
            continue
        n_tr, d = Xtr.shape[0], Xtr.shape[1]
        grid_N = dataset_budget_grid(n_tr)
        print(f"  {name}: n_tr={n_tr}, d={d}, budget grid {grid_N}")

        # -- (a) probe fits: the real rows of the joint-fit table ----------------------
        for kind, k, G, s in itertools.product(("edge_kan", "kanet"), S["orders"],
                                               S["grids"], S["seeds"]):
            if B.exhausted(): break
            key = f"R|probe|{name}|{kind}|k{k}|G{G}|s{s}"
            if key in done: continue
            E = edge_count(kind, d)
            seed_everything(s)
            m = EdgeKAN(d, G, k) if kind == "edge_kan" else KANet(d, G, k)
            r = train_eval(m, (Xtr, ytr, Xte, yte), S["epochs"], s)
            rows.append(dict(key=key, part="probe", dataset=name, kind=kind, k=k, grid=G,
                             seed=s, d=d, n_train=n_tr, edges=E, params=count_params(m),
                             extra=0, mode=None, N=None, **r))

        # -- (b) C5 sign test: error against spurious-edge count -----------------------
        for mode, extra, s in itertools.product(("noise", "duplicate"),
                                                (0, 2, 4, 8, 16, 32), S["seeds"]):
            if B.exhausted(): break
            key = f"R|C5|{name}|{mode}|e{extra}|s{s}"
            if key in done: continue
            Xa, Xb = append_edges(Xtr, Xte, extra, mode, s)
            seed_everything(s)
            mdl = EdgeKAN(Xa.shape[1], 8, 3)
            r = train_eval(mdl, (Xa, ytr, Xb, yte), S["epochs"], s)
            rows.append(dict(key=key, part="C5", dataset=name, kind="edge_kan", k=3, grid=8,
                             seed=s, d=Xa.shape[1], n_train=n_tr,
                             edges=Xa.shape[1], params=count_params(mdl),
                             extra=extra, mode=mode, N=None, **r))

        # -- (c) C6 damage against budget, on the adapted grid -------------------------
        for mode, N, s in itertools.product(("noise", "duplicate"), grid_N, S["seeds"]):
            if B.exhausted(): break
            for extra in (0, 16):
                key = f"R|C6|{name}|{mode}|N{N}|e{extra}|s{s}"
                if key in done: continue
                Xa, Xb = append_edges(Xtr, Xte, extra, mode, s)
                da = Xa.shape[1]
                G = max(2, int(round(N / da - 4)))          # k = 3
                seed_everything(s)
                mdl = EdgeKAN(da, G, 3)
                r = train_eval(mdl, (Xa, ytr, Xb, yte), S["epochs"], s)
                rows.append(dict(key=key, part="C6", dataset=name, kind="edge_kan", k=3,
                                 grid=G, seed=s, d=da, n_train=n_tr, edges=da,
                                 params=count_params(mdl), extra=extra, mode=mode, N=N, **r))

        if rows:
            pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                      header=not os.path.exists(out_csv))
            rows = []
            print(f"    {name} done, {(time.time()-t0)/60:.1f} min elapsed")
    if rows:
        pd.DataFrame(rows).to_csv(out_csv, mode="a", index=False,
                                  header=not os.path.exists(out_csv))
    print(f"R done in {(time.time()-t0)/60:.1f} min")


REAL_CSV = os.path.join(OUT, f"real_{SCALE}.csv")
run_real(S, REAL_CSV)

In [ ]:
REAL = _load(REAL_CSV, "R")

if not REAL.empty:
    # ---- (a) real-data probe fits ---------------------------------------------------
    pr = REAL[REAL.part == "probe"]
    rows = []
    for (ds, kind, k), g in pr.groupby(["dataset", "kind", "k"]):
        cells = (g.groupby(["grid", "params", "edges"])["rel_l2"].mean()
                  .reset_index().rename(columns={"rel_l2": "err"}))
        cells = cells[(cells.err < 0.8) & (cells.err > 0)]
        if len(cells) < 4: continue
        cN, cE, cG = joint_fit(cells, k)
        rows.append(dict(dataset=ds, kind=kind, k=int(k), n_cells=len(cells),
                         cN=round(cN, 3), cE=round(cE, 3), cG=round(cG, 3),
                         cG_pred=-(k + 1)))
    REAL_FIT = pd.DataFrame(rows)
    print("Real-data probe fits (the *Real* rows of the joint-fit table):")
    print(REAL_FIT.to_string(index=False) if len(REAL_FIT) else "  no admissible fits")

    # ---- (b) C5 sign test ------------------------------------------------------------
    c5 = REAL[REAL.part == "C5"]
    rows = []
    for (ds, mode), g in c5.groupby(["dataset", "mode"]):
        ag = g.groupby("extra")["rel_l2"].mean().reset_index().sort_values("extra")
        if len(ag) >= 4:
            rho, p = stats.spearmanr(ag.extra, ag.rel_l2)
            rows.append(dict(dataset=ds, mode=mode, n_points=len(ag),
                             rho=round(float(rho), 3), p=round(float(p), 4),
                             err_at_0=round(float(ag.rel_l2.iloc[0]), 4),
                             err_at_max=round(float(ag.rel_l2.iloc[-1]), 4),
                             trend_present=bool(rho > 0 and p < 0.1)))
    C5 = pd.DataFrame(rows)
    print("\nC5 -- error against spurious-edge count (law predicts rho > 0):")
    print(C5.to_string(index=False) if len(C5) else "  no admissible cells")
    if len(C5) and "duplicate" in set(C5["mode"]):
        dup = C5[C5["mode"] == "duplicate"]
        if (dup.rho < 0).any():
            print("\n  NOTE: the duplicate-edge control shows a NEGATIVE trend on "
                  f"{int((dup.rho < 0).sum())} of {len(dup)} datasets. Duplicating an "
                  "informative feature adds edges but also adds capacity on signal the "
                  "model can already use, so error can fall. Where this happens the "
                  "duplicate arm does not isolate edge count from signal-to-noise and "
                  "should not be read as a control; only the noise arm is interpretable.")

    # ---- (c) C6 damage against budget -------------------------------------------------
    c6 = REAL[REAL.part == "C6"]
    rows = []
    for (ds, mode), g in c6.groupby(["dataset", "mode"]):
        piv = g.groupby(["N", "extra"])["rel_l2"].mean().unstack()
        if not {0, 16}.issubset(piv.columns): continue
        dmg = (piv[16] - piv[0]).dropna()
        if len(dmg) < 3: continue
        rho, p = stats.spearmanr(dmg.index.values, dmg.values)
        # A decay exponent is only defined where the damage is positive; when the
        # spurious edges help rather than hurt, beta is left undefined by design.
        pos = dmg[dmg > 0]
        beta = (float(np.polyfit(np.log(pos.index.values), np.log(pos.values), 1)[0])
                if len(pos) >= 3 else float("nan"))
        rows.append(dict(dataset=ds, mode=mode, n_budgets=len(dmg),
                         rho=round(float(rho), 3), p=round(float(p), 4),
                         beta=round(beta, 3) if beta == beta else None,
                         decay_present=bool(rho < 0 and p < 0.1)))
    C6 = pd.DataFrame(rows)
    print("\nC6 -- spurious-edge damage against budget (law predicts rho < 0):")
    print(C6.to_string(index=False) if len(C6) else "  no admissible cells")
else:
    REAL_FIT = C5 = C6 = pd.DataFrame()

## 4. Part I — integrity checks

The three equivalence checks reported in §4.1 of the manuscript, plus a fourth on the spline
basis itself. These are cheap and run on CPU.

In [ ]:
def check_sizing(dims=(2, 8, 32), budgets=(2_000, 16_000, 128_000), grid=8, order=3):
    """Closed-form width sizing must agree with exhaustive search over widths."""
    agree = total = 0
    for d, N in itertools.product(dims, budgets):
        for a, b, wmax, lab in ((grid + order + 1, d + 4, 1024, "FullKAN"),
                                (2 * 2, 4 * 2 + d + 2, 2048, "MLP")):
            closed = _closest_width(a, b, N, wmax)
            brute = min(range(2, wmax), key=lambda w: abs(a * w * w + b * w + 1 - N))
            agree += int(abs(a*closed**2 + b*closed + 1 - N) <= abs(a*brute**2 + b*brute + 1 - N))
            total += 1
    return agree, total


def check_recompute(n=6, seed_base=0):
    """Re-running a probe cell from its seed must reproduce it exactly."""
    worst = 0.0
    for i in range(n):
        d, G, k, s = (2, 4, 8, 16)[i % 4], (4, 8, 16)[i % 3], 3, seed_base + i
        data = make_synth("additive", d, 2_000, 1_000, s)
        outs = []
        for _ in range(2):
            seed_everything(s)
            outs.append(train_eval(EdgeKAN(d, G, k), data, 5, s)["rel_l2"])
        worst = max(worst, abs(outs[0] - outs[1]))
    return worst


def check_tf32():
    """The TF32 vs float32 discrepancy must sit below the between-seed standard deviation."""
    if DEVICE.type != "cuda":
        return None, None
    data = make_synth("additive", 8, 4_000, 2_000, 0)
    res = {}
    for tf32 in (False, True):
        torch.backends.cuda.matmul.allow_tf32 = tf32
        torch.backends.cudnn.allow_tf32 = tf32
        seed_everything(0)
        res[tf32] = train_eval(full_kan_for_budget(8, 8_000, 8, 3), data, 30, 0)["rel_l2"]
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    seeds = []
    for s in range(3):
        seed_everything(s)
        seeds.append(train_eval(full_kan_for_budget(8, 8_000, 8, 3),
                                make_synth("additive", 8, 4_000, 2_000, s), 30, s)["rel_l2"])
    return abs(res[True] - res[False]), float(np.std(seeds))


def check_partition_of_unity(grids=(4, 8, 16, 32), orders=(2, 3, 4)):
    """The B-spline basis must sum to one across the interior of the domain."""
    worst = 0.0
    for G, k in itertools.product(grids, orders):
        lay = KANLayer(1, 1, grid=G, order=k)
        x = torch.linspace(-0.98, 0.98, 512).unsqueeze(1)
        with torch.no_grad():
            worst = max(worst, float((lay._basis(x).sum(-1) - 1.0).abs().max()))
    return worst


agree, total = check_sizing()
recompute = check_recompute()
tf32_gap, seed_sd = check_tf32()
pou = check_partition_of_unity()

INTEGRITY = dict(
    closed_form_sizing=f"{agree}/{total} configurations agree with exhaustive search",
    probe_recompute=f"max |Delta rel_l2| = {recompute:.2e} across repeated runs",
    tf32_vs_fp32=(f"discrepancy {tf32_gap:.2e} vs between-seed sd {seed_sd:.2e}"
                  if tf32_gap is not None else "not applicable on CPU"),
    partition_of_unity=f"max |sum(basis) - 1| = {pou:.2e}",
    all_pass=bool(agree == total and recompute < 1e-6 and pou < 1e-5
                  and (tf32_gap is None or tf32_gap < seed_sd)),
)
print(json.dumps(INTEGRITY, indent=2))
with open(os.path.join(OUT, "integrity_recomputed.json"), "w") as f:
    json.dump(INTEGRITY, f, indent=2)

## 5. Report

In [ ]:
lines = [f"Completion run, scale = {SCALE}, device = {DEVICE}", "=" * 70, ""]

# ---- E4: the decisive comparison -----------------------------------------------------
if not E4.empty:
    kk = 3 if 3 in E4.k.unique() else int(E4.k.iloc[0])
    sl = dslope_fixed_grid(E4, kk, "rel_l2")
    tr = dslope_fixed_grid(E4, kk, "rel_l2_train")
    mean_test = float(np.nanmean(sl.values)); mean_train = float(np.nanmean(tr.values))
    lines += ["E4 -- WidthKAN, depth 1, edge count E = m*d:",
              f"  dimensional slope at fixed resolution: test {mean_test:+.2f}, "
              f"train {mean_train:+.2f}",
              "  reference values from the paper and from E1:",
              "    EdgeKAN (depth 1, E = d)          about  -0.2  (premise satisfied)",
              "    closed-form oracle                       0.00 to 0.02",
              "    KANet   (depth 2, E ~ 2d^2)       about  +0.9 test, +0.7 train", ""]
    if abs(mean_test) < 0.3:
        lines += ["  READING: WidthKAN behaves like EdgeKAN. Raising the edge count at fixed",
                  "  depth does NOT reproduce the dimensional slope, so the effect is specific",
                  "  to the two-layer COMPOSITION rather than to edge count. Section 5.2 can",
                  "  name depth as the remaining candidate mechanism."]
    elif mean_test > 0.6:
        lines += ["  READING: WidthKAN behaves like KANet. The dimensional slope follows the",
                  "  EDGE COUNT and not depth. This removes depth as a confound and points the",
                  "  mechanism back at edge count -- which the edge law predicts, but with the",
                  "  wrong sign on the budget coefficient, so Section 5.2 must reconcile the two."]
    else:
        lines += ["  READING: WidthKAN sits between EdgeKAN and KANet. Depth and edge count",
                  "  both appear to contribute; neither alone explains the effect."]
    lines.append("")
    if not E4_FIT.empty:
        for r in E4_FIT.itertuples():
            lines.append(f"  joint fit k={r.k}: c_E {r.cE:+.3f} (pred {r.cE_pred}) | "
                         f"c_G {r.cG:+.3f} (pred {r.cG_pred}, pass={r.cG_pass})")
        lines.append("")
else:
    lines += ["E4 -- no cells completed.", ""]

# ---- R --------------------------------------------------------------------------------
if not REAL.empty:
    lines += ["R -- real data:",
              f"  probe fits admissible for {len(REAL_FIT)} (dataset, probe, order) combinations"]
    if len(C5):
        ok = int(C5.trend_present.sum())
        lines += [f"  C5 sign test: trend present in {ok}/{len(C5)} dataset-mode cells "
                  f"(paper: 1 of 3 real datasets returned a statistic at all)"]
    if len(C6):
        ok = int(C6.decay_present.sum())
        lines += [f"  C6 damage decay: present in {ok}/{len(C6)} dataset-mode cells "
                  f"(paper: no real dataset yielded an estimate)"]
    lines += ["  The budget grid is now derived from each dataset's own training split, which",
              "  is what the original fixed grid got wrong.", ""]
else:
    lines += ["R -- no rows produced.", ""]

# ---- I --------------------------------------------------------------------------------
lines += ["I -- integrity checks:"]
for k, v in INTEGRITY.items():
    if k != "all_pass": lines.append(f"  {k}: {v}")
lines += [f"  all pass: {INTEGRITY['all_pass']}", ""]

lines += ["Scope: E4 and the real-data pipeline were named as gaps in the manuscript and are",
          "reported as executed after the pre-registered study. None of this revises the",
          "pre-registered verdict.", "",
          "=" * 70,
          "DOWNLOAD THESE BEFORE CLOSING THE SESSION", "=" * 70]
for _f in sorted(os.listdir(OUT)):
    _p = os.path.join(OUT, _f)
    if os.path.isfile(_p):
        lines.append(f"  {os.path.getsize(_p):>10,} B   {_p}")

report = "\n".join(lines)
print(report)

# Offer the two result files as direct downloads, so nothing depends on finding them later.
try:
    from google.colab import files as _colab_files
    for _f in (E4_CSV, REAL_CSV):
        if os.path.exists(_f):
            print(f"downloading {os.path.basename(_f)} ...")
            _colab_files.download(_f)
except Exception as _e:
    print(f"(automatic download unavailable: {type(_e).__name__} -- "
          f"copy the files listed above by hand)")
with open(os.path.join(OUT, f"completion_summary_{SCALE}.txt"), "w") as f:
    f.write(report)
print("\nwritten:", sorted(os.listdir(OUT)))